In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the absolute data path
DATA_PATH = "/Users/darshanv/credit-risk-scorer/data/"

print("Loading datasets... this might take a minute.\n")

# 1. Main Application Table (Static Snapshot)
app_train = pd.read_csv(f"{DATA_PATH}application_train.csv")
print(f"Main Application (app_train): {app_train.shape}")

# 2. Supplementary Tables (Behavioral History)
bureau = pd.read_csv(f"{DATA_PATH}bureau.csv")
print(f"Bureau (bureau): {bureau.shape}")

bureau_balance = pd.read_csv(f"{DATA_PATH}bureau_balance.csv")
print(f"Bureau Balance (bureau_balance): {bureau_balance.shape}")

prev_app = pd.read_csv(f"{DATA_PATH}previous_application.csv")
print(f"Previous Applications (prev_app): {prev_app.shape}")

installments = pd.read_csv(f"{DATA_PATH}installments_payments.csv")
print(f"Installments Payments (installments): {installments.shape}")

cc_balance = pd.read_csv(f"{DATA_PATH}credit_card_balance.csv")
print(f"Credit Card Balance (cc_balance): {cc_balance.shape}")

Loading datasets... this might take a minute.

Main Application (app_train): (307511, 122)
Bureau (bureau): (1716428, 17)
Bureau Balance (bureau_balance): (27299925, 3)
Previous Applications (prev_app): (1670214, 37)
Installments Payments (installments): (13605401, 8)
Credit Card Balance (cc_balance): (3840312, 23)


In [2]:
print("Aggregating bureau.csv...")

# 1. Base mathematical aggregations for each applicant
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_loan_count=('SK_ID_BUREAU', 'count'),
    bureau_avg_days_credit=('DAYS_CREDIT', 'mean'),
    bureau_avg_credit_day_overdue=('CREDIT_DAY_OVERDUE', 'mean'),
    bureau_max_days_overdue=('CREDIT_DAY_OVERDUE', 'max'),
    bureau_total_debt=('AMT_CREDIT_SUM_DEBT', 'sum'),
    total_credit_limit=('AMT_CREDIT_SUM', 'sum') # Helper column for the ratio
).reset_index()

# 2. Count 'Active' and 'Closed' loans specifically
active_counts = bureau[bureau['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR').size().reset_index(name='bureau_active_loans')
closed_counts = bureau[bureau['CREDIT_ACTIVE'] == 'Closed'].groupby('SK_ID_CURR').size().reset_index(name='bureau_closed_loans')

# 3. Merge the specific counts back into our main aggregated dataframe
bureau_agg = bureau_agg.merge(active_counts, on='SK_ID_CURR', how='left')
bureau_agg = bureau_agg.merge(closed_counts, on='SK_ID_CURR', how='left')

# Fill NaNs with 0 (if an applicant has no active loans, the count is 0, not missing)
bureau_agg['bureau_active_loans'] = bureau_agg['bureau_active_loans'].fillna(0)
bureau_agg['bureau_closed_loans'] = bureau_agg['bureau_closed_loans'].fillna(0)

# 4. Calculate the Debt-to-Credit Ratio
# We add a tiny number (1e-5) to the denominator to mathematically prevent "Division by Zero" errors
bureau_agg['bureau_debt_to_credit_ratio'] = bureau_agg['bureau_total_debt'] / (bureau_agg['total_credit_limit'] + 1e-5)

# Drop the helper column as we only needed it for the math
bureau_agg = bureau_agg.drop(columns=['total_credit_limit'])

print(f"Bureau Aggregation Complete. Shape: {bureau_agg.shape}")
display(bureau_agg.head())

Aggregating bureau.csv...
Bureau Aggregation Complete. Shape: (305811, 9)


,SK_ID_CURR,bureau_loan_count,bureau_avg_days_credit,bureau_avg_credit_day_overdue,bureau_max_days_overdue,bureau_total_debt,bureau_active_loans,bureau_closed_loans,bureau_debt_to_credit_ratio
0,100001,7,-735.000000,0.0,0,596686.5,3.0,4.0,0.410555
1,100002,8,-874.000000,0.0,0,245781.0,2.0,6.0,0.284122
2,100003,4,-1400.750000,0.0,0,0.0,1.0,3.0,0.000000
3,100004,2,-867.000000,0.0,0,0.0,0.0,2.0,0.000000
4,100005,3,-190.666667,0.0,0,568408.5,2.0,1.0,0.864992


In [3]:
print("Aggregating bureau_balance.csv... (This requires a two-step rollup)")

# 1. Map the string status codes to numeric risk scores
# 'C' (Closed), 'X' (Unknown), '0' (No delay) represent zero active risk.
# '1' through '5' represent increasing severity of days past due.
status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bureau_balance['STATUS_NUM'] = bureau_balance['STATUS'].map(status_map)

# Create a binary flag for any month where they were late (status 1-5)
bureau_balance['IS_LATE'] = bureau_balance['STATUS'].isin(['1', '2', '3', '4', '5']).astype(int)

# 2. Step One: Roll up to the LOAN level (SK_ID_BUREAU)
bb_loan_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(
    bureau_bal_total_records=('MONTHS_BALANCE', 'count'),
    bureau_bal_bad_count=('IS_LATE', 'sum'),
    bureau_bal_avg_status=('STATUS_NUM', 'mean')
).reset_index()

# 3. Connect the loans to the applicants
# We pull SK_ID_CURR from the original bureau dataframe
bb_merged = bb_loan_agg.merge(bureau[['SK_ID_BUREAU', 'SK_ID_CURR']], on='SK_ID_BUREAU', how='inner')

# 4. Step Two: Roll up to the APPLICANT level (SK_ID_CURR)
bureau_bal_final = bb_merged.groupby('SK_ID_CURR').agg(
    bureau_bal_total_records=('bureau_bal_total_records', 'sum'),
    bureau_bal_bad_count=('bureau_bal_bad_count', 'sum'),
    bureau_bal_avg_status=('bureau_bal_avg_status', 'mean') 
).reset_index()

# 5. Calculate the overall bad ratio for the applicant
bureau_bal_final['bureau_bal_bad_ratio'] = bureau_bal_final['bureau_bal_bad_count'] / (bureau_bal_final['bureau_bal_total_records'] + 1e-5)

print(f"Bureau Balance Aggregation Complete. Shape: {bureau_bal_final.shape}")
display(bureau_bal_final.head())

Aggregating bureau_balance.csv... (This requires a two-step rollup)
Bureau Balance Aggregation Complete. Shape: (134542, 5)


,SK_ID_CURR,bureau_bal_total_records,bureau_bal_bad_count,bureau_bal_avg_status,bureau_bal_bad_ratio
0,100001,172,1,0.007519,0.005814
1,100002,110,27,0.255682,0.245455
2,100005,21,0,0.000000,0.000000
3,100010,72,0,0.000000,0.000000
4,100013,230,7,0.027701,0.030435


In [5]:
print("Aggregating previous_application.csv...")

# 1. Create binary flags for the application statuses we care about
# (Note: The dataset spells it 'Canceled', but we check both just to be safe)
prev_app['IS_APPROVED'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
prev_app['IS_REFUSED'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
prev_app['IS_CANCELLED'] = prev_app['NAME_CONTRACT_STATUS'].isin(['Canceled', 'Cancelled']).astype(int)

# 2. Roll up to the Applicant level (SK_ID_CURR)
prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    prev_app_count=('SK_ID_PREV', 'count'),
    prev_approved_count=('IS_APPROVED', 'sum'),
    prev_refused_count=('IS_REFUSED', 'sum'),
    prev_cancelled_count=('IS_CANCELLED', 'sum'),
    prev_avg_credit=('AMT_APPLICATION', 'mean'), 
    prev_avg_down_payment=('AMT_DOWN_PAYMENT', 'mean'),
    prev_avg_days_decision=('DAYS_DECISION', 'mean')
).reset_index()

# 3. Calculate the Approval Rate
# Adding 1e-5 to prevent division by zero
prev_agg['prev_approval_rate'] = prev_agg['prev_approved_count'] / (prev_agg['prev_app_count'] + 1e-5)

print(f"Previous Applications Aggregation Complete. Shape: {prev_agg.shape}")
display(prev_agg.head())

Aggregating previous_application.csv...
Previous Applications Aggregation Complete. Shape: (338857, 9)


,SK_ID_CURR,prev_app_count,prev_approved_count,prev_refused_count,prev_cancelled_count,prev_avg_credit,prev_avg_down_payment,prev_avg_days_decision,prev_approval_rate
0,100001,1,1,0,0,24835.50,2520.0,-1740.0,0.999990
1,100002,1,1,0,0,179055.00,0.0,-606.0,0.999990
2,100003,3,3,0,0,435436.50,3442.5,-1305.0,0.999997
3,100004,1,1,0,0,24282.00,4860.0,-815.0,0.999990
4,100005,2,1,0,1,22308.75,4464.0,-536.0,0.499998


In [7]:
print("Aggregating installments_payments.csv...")

# 1. Calculate how late each payment was (Days Late)
# The dates are negative (e.g., -15 means due 15 days ago). 
# If due -15 and paid -10, that's 5 days late: (-10) - (-15) = 5
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']

# If they paid early (negative days late), we just count it as 0 days late to keep the math clean.
installments['DAYS_LATE'] = installments['DAYS_LATE'].apply(lambda x: x if x > 0 else 0)
installments['IS_LATE'] = (installments['DAYS_LATE'] > 0).astype(int)

# 2. Calculate the Payment Ratio (Did they pay the full amount?)
# Adding 1e-5 to prevent division by zero
installments['PAYMENT_RATIO'] = installments['AMT_PAYMENT'] / (installments['AMT_INSTALMENT'] + 1e-5)

# 3. Roll up to the Applicant level (SK_ID_CURR)
instal_agg = installments.groupby('SK_ID_CURR').agg(
    instal_count=('SK_ID_PREV', 'count'),  # Total payments made
    instal_avg_days_late=('DAYS_LATE', 'mean'),
    instal_max_days_late=('DAYS_LATE', 'max'),
    instal_late_payment_count=('IS_LATE', 'sum'),
    instal_avg_payment_ratio=('PAYMENT_RATIO', 'mean')
).reset_index()

# 4. Calculate the fraction of payments made late
instal_agg['instal_late_payment_ratio'] = instal_agg['instal_late_payment_count'] / (instal_agg['instal_count'] + 1e-5)

print(f"Installments Aggregation Complete. Shape: {instal_agg.shape}")
display(instal_agg.head())

Aggregating installments_payments.csv...
Installments Aggregation Complete. Shape: (339587, 7)


,SK_ID_CURR,instal_count,instal_avg_days_late,instal_max_days_late,instal_late_payment_count,instal_avg_payment_ratio,instal_late_payment_ratio
0,100001,7,1.571429,11.0,1,1.0,0.142857
1,100002,19,0.000000,0.0,0,1.0,0.000000
2,100003,25,0.000000,0.0,0,1.0,0.000000
3,100004,3,0.000000,0.0,0,1.0,0.000000
4,100005,9,0.111111,1.0,1,1.0,0.111111


In [8]:
print("Aggregating credit_card_balance.csv...")

# 1. Calculate Utilization (How much of their limit are they using?)
# Add 1e-5 to prevent division by zero
cc_balance['CC_UTILIZATION'] = cc_balance['AMT_BALANCE'] / (cc_balance['AMT_CREDIT_LIMIT_ACTUAL'] + 1e-5)

# 2. Calculate Payment Ratio (Are they paying more than the minimum?)
# Add 1e-5 to prevent division by zero
cc_balance['CC_PAYMENT_RATIO'] = cc_balance['AMT_PAYMENT_CURRENT'] / (cc_balance['AMT_INST_MIN_REGULARITY'] + 1e-5)

# 3. Create a binary flag for late payments (SK_DPD = Days Past Due)
cc_balance['IS_LATE'] = (cc_balance['SK_DPD'] > 0).astype(int)

# 4. Roll up to the Applicant level (SK_ID_CURR)
cc_agg = cc_balance.groupby('SK_ID_CURR').agg(
    cc_total_records=('MONTHS_BALANCE', 'count'),
    cc_avg_balance=('AMT_BALANCE', 'mean'),
    cc_max_balance=('AMT_BALANCE', 'max'),
    cc_avg_utilization=('CC_UTILIZATION', 'mean'),
    cc_max_utilization=('CC_UTILIZATION', 'max'),
    cc_avg_payment_ratio=('CC_PAYMENT_RATIO', 'mean'),
    cc_late_payment_count=('IS_LATE', 'sum')
).reset_index()

print(f"Credit Card Aggregation Complete. Shape: {cc_agg.shape}")
display(cc_agg.head())

Aggregating credit_card_balance.csv...
Credit Card Aggregation Complete. Shape: (103558, 8)


,SK_ID_CURR,cc_total_records,cc_avg_balance,cc_max_balance,cc_avg_utilization,cc_max_utilization,cc_avg_payment_ratio,cc_late_payment_count
0,100006,6,0.000000,0.00,0.000000,0.00000,NaN,0
1,100011,74,54482.111149,189000.00,0.302678,1.05000,3.086877e+07,0
2,100013,96,18159.919219,161420.22,0.115301,1.02489,2.652497e+07,1
3,100021,17,0.000000,0.00,0.000000,0.00000,NaN,0
4,100023,8,0.000000,0.00,0.000000,0.00000,NaN,0


In [9]:
print("Merging all tables into the Master Dataframe...")

# 1. Start with the main application table (307,511 rows)
master_df = app_train.copy()

# 2. Left join all aggregated tables one by one using the applicant ID
master_df = master_df.merge(bureau_agg, on='SK_ID_CURR', how='left')
master_df = master_df.merge(bureau_bal_final, on='SK_ID_CURR', how='left')
master_df = master_df.merge(prev_agg, on='SK_ID_CURR', how='left')
master_df = master_df.merge(instal_agg, on='SK_ID_CURR', how='left')
master_df = master_df.merge(cc_agg, on='SK_ID_CURR', how='left')

print(f"Original app_train shape: {app_train.shape}")
print(f"New Master Dataframe shape: {master_df.shape}")

# Let's peek at the newly added columns for the first 5 rows
new_cols = list(bureau_agg.columns) + list(bureau_bal_final.columns) + \
           list(prev_agg.columns) + list(instal_agg.columns) + list(cc_agg.columns)
# Remove duplicate SK_ID_CURR from the list
new_cols = [col for col in set(new_cols) if col != 'SK_ID_CURR']

display(master_df[['SK_ID_CURR'] + new_cols[:10]].head())

Merging all tables into the Master Dataframe...
Original app_train shape: (307511, 122)
New Master Dataframe shape: (307511, 155)


,SK_ID_CURR,prev_approved_count,prev_avg_down_payment,cc_max_balance,bureau_avg_days_credit,bureau_active_loans,bureau_avg_credit_day_overdue,cc_max_utilization,instal_late_payment_count,bureau_closed_loans,bureau_bal_avg_status
0,100002,1.0,0.00,NaN,-874.00,2.0,0.0,NaN,0.0,6.0,0.255682
1,100003,3.0,3442.50,NaN,-1400.75,1.0,0.0,NaN,0.0,3.0,NaN
2,100004,1.0,4860.00,NaN,-867.00,0.0,0.0,NaN,0.0,2.0,NaN
3,100006,5.0,34840.17,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN
4,100007,6.0,3390.75,NaN,-1149.00,0.0,0.0,NaN,16.0,1.0,NaN


In [10]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

print("Starting Preprocessing v2...")

# 1. Fix Anomalies
master_df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# 2. Recreate Phase 4 Engineered Features
master_df['age_years'] = -master_df['DAYS_BIRTH'] / 365
master_df['income_annuity_ratio'] = master_df['AMT_INCOME_TOTAL'] / master_df['AMT_ANNUITY']
master_df['employment_to_age_ratio'] = -master_df['DAYS_EMPLOYED'] / -master_df['DAYS_BIRTH']
master_df['null_count'] = master_df.isnull().sum(axis=1)
master_df['credit_to_income_ratio'] = master_df['AMT_CREDIT'] / master_df['AMT_INCOME_TOTAL']

# 3. Handle Behavioral NaNs (The "Ghosts")
# We find all columns we just created using their prefixes and fill missing with 0
behavioral_prefixes = ['bureau_', 'prev_', 'instal_', 'cc_']
behavioral_cols = [col for col in master_df.columns if any(col.startswith(p) for p in behavioral_prefixes)]
master_df[behavioral_cols] = master_df[behavioral_cols].fillna(0)

# 4. Drop high-missing columns (excluding EXT_SOURCE)
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
missing_pct = master_df.isnull().sum() / len(master_df)
cols_to_drop = missing_pct[(missing_pct > 0.40) & (~missing_pct.index.isin(ext_sources))].index
master_df.drop(columns=cols_to_drop, inplace=True)

# 5. General Imputation for the remaining columns
numeric_cols = master_df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = master_df.select_dtypes(include=['object']).columns

# Median for numerics
for col in numeric_cols:
    if master_df[col].isnull().any():
        master_df[col].fillna(master_df[col].median(), inplace=True)
        
# Mode for categoricals
for col in categorical_cols:
    if master_df[col].isnull().any():
        master_df[col].fillna(master_df[col].mode()[0], inplace=True)
        
# 6. Label Encoding for categoricals
le = LabelEncoder()
for col in categorical_cols:
    master_df[col] = le.fit_transform(master_df[col].astype(str))
    
# 7. Drop ID and Save
master_df.drop(columns=['SK_ID_CURR'], inplace=True)

save_path = "/Users/darshanv/credit-risk-scorer/data/application_train_processed_v2.csv"
master_df.to_csv(save_path, index=False)

print(f"Preprocessing Complete. Final shape for modeling: {master_df.shape}")
print(f"Saved to: {save_path}")

Starting Preprocessing v2...
Preprocessing Complete. Final shape for modeling: (307511, 111)
Saved to: /Users/darshanv/credit-risk-scorer/data/application_train_processed_v2.csv
